En esta sección se realiza una limpieza de los datos para corregir los errores detectados en la sección de Análisis Exploratorio de Datos. Se corrigen los errores en los identificadores del dataset y se renombran los archivos que contienen espacios o que no siguen el formato establecido.

In [1]:
from src.vasijas.reporting import ProcessingReport
from src.vasijas.files import normalizar_nombre
from _bootstrap import setup
from pathlib import Path
import shutil
import pandas as pd

PATHS = setup()
report = ProcessingReport()

Normalización de los nombres de los archivos:
* minúsculas
* sin acentos
* caracteres no alfanuméricos -> _
* colapsa múltiples _
* conserva extensión

In [2]:
ruta_ubicaciones: list[Path] = [
    ruta
    for ruta in PATHS.DATA_ORIGINALES.iterdir()
    if ruta.is_dir()
]
for ruta_ubicacion in ruta_ubicaciones:
    nombre_ubicacion = ruta_ubicacion.name

    ruta_destino = PATHS.DATA_CORREGIDOS / nombre_ubicacion
    ruta_destino.mkdir(parents=True, exist_ok=True)

    archivos_png = list(ruta_ubicacion.glob("*.png"))

    for archivo in archivos_png:
        nombre_normalizado = normalizar_nombre(archivo.name)
        destino = ruta_destino / nombre_normalizado
        shutil.copy2(archivo, destino)

Creación de un nuevo archivo de metadata con Identificadores normalizados:
* minúsculas
* sin acentos
* caracteres no alfanuméricos -> _
* colapsa múltiples _
* conserva extensión

In [3]:
csv_origen: Path = PATHS.DATA_ORIGINALES / "metadata.csv"
csv_destino: Path = PATHS.DATA_CORREGIDOS / "metadata.csv"

df = pd.read_csv(csv_origen)
df["Identificador"] = (
    df["Identificador"]
    .astype(str)
    .apply(normalizar_nombre)
)
df.head()

,Identificador,Tiene asa,Tendencia Formal,Borde,Cuello,Cuerpo,Base,Clase Ana,Diametro Borde,Diametro Base,Altura Total,Cronologia
0,al_03a,No,II,1,B,2,2,11.0,182,54,53,ibero
1,al_03b,No,II,1,B,2,2,11.0,178,57,46,ibero
2,al_03c,No,II,1,B,2,2,11.0,185,56,49,ibero
3,al_03d,No,II,1,B,2,2,11.0,160,41,45,ibero
4,al_03e,No,II,1,B,2,2,11.0,164,54,44,ibero


In [4]:
# Identificadores del dataset que aparecen en los nombres de los archivos
nombres_archivos_png: set[str] = {p.stem for p in PATHS.DATA_CORREGIDOS.rglob("*.png")}

df["id_encontrado_en_fs"] = (
    df["Identificador"]
    .astype(str)
    .str.strip()
    .isin(nombres_archivos_png)
)

cantidad_encontrados = int(df["id_encontrado_en_fs"].sum())
cantidad_no_encontrados = int((~df["id_encontrado_en_fs"]).sum())

print(f"Cantidad de Identificadores que aparecen en los nombres de los archivos: {cantidad_encontrados}")
print(f"Cantidad de Identificadores que NO aparecen en los nombres de los archivos: {cantidad_no_encontrados}\n")

report.add_metric("cant_identificadores_coinciden_con_nombre_archivos", cantidad_encontrados)
report.add_metric("cant_identificadores_no_coinciden_con_nombre_archivos", cantidad_no_encontrados)

Cantidad de Identificadores que aparecen en los nombres de los archivos: 1240
Cantidad de Identificadores que NO aparecen en los nombres de los archivos: 2



In [5]:
# Detalle de Identificadores del dataset que no encontrados en el conjunto de nombres de archivos:

ids_no_encontrados = set(df.loc[~df["id_encontrado_en_fs"], "Identificador"].astype(str).str.strip())
cantidad_no_encontrados = len(ids_no_encontrados)
ids_no_encontrados_ordenados = sorted(ids_no_encontrados)

print("Cant. de Identificadores que aparecen en el metadata que no se corresponde con el nombre de un archivo PNG:",
      cantidad_no_encontrados)
print("Detalle de identificadores no encontrados en archivos PNG:")
print(ids_no_encontrados_ordenados)

report.add_metric("cant_ids_no_aparecen_entre_nombres_archivos", cantidad_no_encontrados)
report.add_metric("ids_no_aparecen_entre_nombres_archivos", ids_no_encontrados_ordenados)


Cant. de Identificadores que aparecen en el metadata que no se corresponde con el nombre de un archivo PNG: 2
Detalle de identificadores no encontrados en archivos PNG:
['bo_3_2_1', 'cc_0_8_2']


In [6]:
# Archivos PNG sin registro en dataset

ids_en_df = set(df["Identificador"].astype(str).str.strip())

archivos_sin_registro = nombres_archivos_png - ids_en_df
cantidad_archivos_sin_registro = len(archivos_sin_registro)

archivos_sin_registro_ordenados = sorted(archivos_sin_registro)

print(f"Cantidad de archivos PNG sin registro en metadata: {cantidad_archivos_sin_registro}")
print("Detalle de archivos PNG no encontrados en metadata.csv:")
print(archivos_sin_registro_ordenados)

report.add_metric("cant_archivos_png_sin_registro_en_metadata", cantidad_archivos_sin_registro)
report.add_metric("archivos_png_sin_registro_en_metadata", archivos_sin_registro_ordenados)

Cantidad de archivos PNG sin registro en metadata: 2
Detalle de archivos PNG no encontrados en metadata.csv:
['bo_32_1', 'cc_08_2']


Dado que no fueron encontrados archivos PNG correspondiente a los identificadores del archivo metadata.csv:
* bo_3_2_1
* cc_0_8_2

Y que no tienen su correspondiente registro en el archivo metadata.csv los archivos:

* bo_32_1
* cc_08_2

Por la similitud de los nombres se asume que hay una correspondencia entre ellos.

Dando por válidos los nombres de los archivos se procede a renombrar los identificadores:

* bo_3_2_1 -> bo_32_1
* cc_0_8_2 -> cc_08_2


In [7]:
# Corregir identificadores con errores de formato
correcciones = {
    "bo_3_2_1": "bo_32_1",
    "cc_0_8_2": "cc_08_2",
}

df["Identificador"] = df["Identificador"].replace(correcciones)

for original, corregido in correcciones.items():
    report.add_metric(f"id_corregido_{original}", corregido)

print("Identificadores corregidos y metadata guardada.")
df.head()

Identificadores corregidos y metadata guardada.


,Identificador,Tiene asa,Tendencia Formal,Borde,Cuello,Cuerpo,Base,Clase Ana,Diametro Borde,Diametro Base,Altura Total,Cronologia,id_encontrado_en_fs
0,al_03a,No,II,1,B,2,2,11.0,182,54,53,ibero,True
1,al_03b,No,II,1,B,2,2,11.0,178,57,46,ibero,True
2,al_03c,No,II,1,B,2,2,11.0,185,56,49,ibero,True
3,al_03d,No,II,1,B,2,2,11.0,160,41,45,ibero,True
4,al_03e,No,II,1,B,2,2,11.0,164,54,44,ibero,True


In [8]:
df.drop(columns=["id_encontrado_en_fs"], inplace=True)
df.to_csv(csv_destino, index=False)